In [1]:
from ultralytics import YOLO
import cv2
from datetime import datetime

import os
import zipfile
import random
import pandas as pd
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import cv2 # 이미지를 읽기 위해
from pathlib import Path
from tqdm.notebook import tqdm # 진행률 표시줄의 경우
import csv # CSV 인용 옵션의 경우
import shutil

In [2]:
# .pt 파일 생성
# 1. data.yaml 파일 생성 (따로 생성함)
# 2. 라벨 데이터 만들기
# 이미지 자체가 하나의 객체이기 때문에 클래스 파일을 만들어서 분류하면 됨
# 경로 설정(상대 경로)
base_dir = r"./Human_Action_Recognition/"
train_csv = os.path.join(base_dir, "Training_set.csv")
test_csv = os.path.join(base_dir, "Testing_set.csv")

# 이미지 경로
train_img_dir = os.path.join(base_dir, "train")
test_img_dir = os.path.join(base_dir, "test")

# 출력 폴더 (동일 위치에 생성)
output_base = os.path.join(base_dir, "classification_dataset")

# CSV 불러오기
train_df = pd.read_csv(train_csv)
test_df = pd.read_csv(test_csv)

#  컬럼명 공백 제거
train_df.columns = train_df.columns.str.strip()
test_df.columns = test_df.columns.str.strip()

# 결측치 제거 (label 열 기준)
train_df = train_df.dropna(subset=['label'])

In [3]:
# train/val 분할 (클래스 비율 유지)
train_split, val_split = train_test_split(
    train_df,
    test_size=0.2,
    stratify=train_df['label'],
    random_state=42
)

In [4]:
#이미지 복사 함수 정의
def copy_images(split_df, split_name, source_dir):
    for _, row in split_df.iterrows():
        label = row['label']  # 클래스 이름
        filename = row['filename']
        src = os.path.join(source_dir, filename)
        dst_dir = os.path.join(output_base, split_name, label)
        dst = os.path.join(dst_dir, filename)
        os.makedirs(dst_dir, exist_ok=True)
        if os.path.exists(src):
            shutil.copy(src, dst)
        else:
            print(f" 이미지 없음: {src}")

In [5]:
# 테스트용 복사 함수 (label 없음)
def copy_test_images(test_df, source_dir):
    for _, row in test_df.iterrows():
        filename = row['filename']
        src = os.path.join(source_dir, filename)
        dst_dir = os.path.join(output_base, "test")  # label 없음
        dst = os.path.join(dst_dir, filename)
        os.makedirs(dst_dir, exist_ok=True)
        if os.path.exists(src):
            shutil.copy(src, dst)
        else:
            print(f" 테스트 이미지 없음: {src}")

In [6]:
# 실제 복사 실행
copy_images(train_split, "train", train_img_dir)
copy_images(val_split, "val", train_img_dir)
copy_test_images(test_df, test_img_dir)

In [9]:
# 분류모델 불러오기
model = YOLO('yolov8n-cls.pt')  

# 학습 시작
model.train(
    data='./Human_Action_Recognition/classification_dataset',  # 폴더 경로만 주면 돼요!
    epochs=50,
    imgsz=224,  # 일반적으로 224 권장
    batch=32
)

New https://pypi.org/project/ultralytics/8.3.158 available  Update with 'pip install -U ultralytics'
Ultralytics 8.3.153  Python-3.8.0 torch-2.4.1+cu118 CUDA:0 (NVIDIA GeForce RTX 3050 Ti Laptop GPU, 4096MiB)
engine\trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=./Human_Action_Recognition/classification_dataset, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=224, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n-cls.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=tr

train: Scanning C:\Users\USER\graduate2\Human_Action_Recognition\classification_dataset\train... 10080 images, 0 corrup


train: New cache created: C:\Users\USER\graduate2\Human_Action_Recognition\classification_dataset\train.cache
val: Fast image access  (ping: 0.20.0 ms, read: 1.00.4 MB/s, size: 15.5 KB)


val: Scanning C:\Users\USER\graduate2\Human_Action_Recognition\classification_dataset\val... 2520 images, 0 corrupt: 10

val: New cache created: C:\Users\USER\graduate2\Human_Action_Recognition\classification_dataset\val.cache


optimizer: 'optimizer=auto' found, ignoring 'lr0=0.01' and 'momentum=0.937' and determining best 'optimizer', 'lr0' and 'momentum' automatically... 
optimizer: AdamW(lr=0.000526, momentum=0.9) with parameter groups 26 weight(decay=0.0), 27 weight(decay=0.0005), 27 bias(decay=0.0)
Image sizes 224 train, 224 val
Using 8 dataloader workers
Logging results to runs\classify\train2
Starting training for 50 epochs...

      Epoch    GPU_mem       loss  Instances       Size


... (149 outputs left out)


50 epochs completed in 0.287 hours.
Optimizer stripped from runs\classify\train2\weights\last.pt, 3.0MB
Optimizer stripped from runs\classify\train2\weights\best.pt, 3.0MB

Validating runs\classify\train2\weights\best.pt...
Ultralytics 8.3.153  Python-3.8.0 torch-2.4.1+cu118 CUDA:0 (NVIDIA GeForce RTX 3050 Ti Laptop GPU, 4096MiB)
YOLOv8n-cls summary (fused): 30 layers, 1,454,095 parameters, 0 gradients, 3.3 GFLOPs
train: C:\Users\USER\graduate2\Human_Action_Recognition\classification_dataset\train... found 10080 images in 15 classes  
val: C:\Users\USER\graduate2\Human_Action_Recognition\classification_dataset\val... found 2520 images in 15 classes  
ERROR test: C:\Users\USER\graduate2\Human_Action_Recognition\classification_dataset\test... found 5400 images in 1 classes (requires 15 classes, not 1)


               classes   top1_acc   top5_acc: 100%|██████████| 40/40 [00:01<00:00, 22.91it/s]


                   all      0.788      0.972
Speed: 0.2ms preprocess, 0.2ms inference, 0.0ms loss, 0.0ms postprocess per image
Results saved to runs\classify\train2


ultralytics.utils.metrics.ClassifyMetrics object with attributes:

confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x0000021F9B280910>
curves: []
curves_results: []
fitness: 0.879960298538208
keys: ['metrics/accuracy_top1', 'metrics/accuracy_top5']
results_dict: {'metrics/accuracy_top1': 0.7876983880996704, 'metrics/accuracy_top5': 0.9722222089767456, 'fitness': 0.879960298538208}
save_dir: WindowsPath('runs/classify/train2')
speed: {'preprocess': 0.20395103174586265, 'inference': 0.21980075396817889, 'loss': 0.0016974603172890228, 'postprocess': 0.0010211904763100248}
task: 'classify'
top1: 0.7876983880996704
top5: 0.9722222089767456

In [6]:
# 테스트셋 평가
model = YOLO("runs/classify/train/weights/best.pt")  # 학습된 모델 로드

In [ ]:
# 예측 수행 및 결과 저장
results = model("./Human_Action_Recognition/test")
# results = model(frame)
results[0].plot()  # 바운딩 박스가 그려진 이미지



WARNING  inference results will accumulate in RAM unless `stream=True` is passed, causing potential out-of-memory
errors for large sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict/ for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs

image 1/5410 C:\Users\USER\graduate2\Human_Action_Recognition\test\Image_1.jpg: 224x224 sleeping 0.97, drinking 0.02, using_laptop 0.00, calling 0.00, texting 0.00, 61.5ms
image 2/5410 C:\Users\USER\graduate2\Human_Action_Recognition\test\Image_10.jpg: 224x224 drinking 1.00, eating 0.00, laughing 0.00, listening_to_music 0.00, calling 0.00, 13.5ms
image 3/5410 C:\Users\USER\graduate2\Human_Action_Recognition\test\Image_100.jpg: 224x224 running 1.00, cycl

# 웹캠 연결

In [7]:
from ultralytics import YOLO
from deep_sort_realtime.deepsort_tracker import DeepSort
import face_recognition
import cv2
import time
import csv
import os
from datetime import datetime

단지 웹캠을 연결해서 실시간으로 행동을 분류하는지 확인하는 코드

In [3]:
# # 1. 모델 불러오기
# det_model = YOLO('yolov8n.pt')      # 탐지 (사람 등)
# cls_model = YOLO('runs/classify/train/weights/best.pt')     # 분류 (행동 분류)

# # 2. 웹캠 열기
# cap = cv2.VideoCapture(0)

# # 3. CSV 파일 준비
# csv_filename = "realtime_behavior_log.csv"
# if not os.path.exists(csv_filename):
#     with open(csv_filename, mode='w', newline='') as f:
#         writer = csv.writer(f)
#         writer.writerow(["timestamp", "object", "behavior", "confidence", "box"])

# # 4. 타이머 초기화
# start_time = time.time()

# while True:
#     ret, frame = cap.read()
#     if not ret:
#         break

#     # 프레임 저장 디렉토리 준비
#     frame_save_dir = "saved_frames"
#     os.makedirs(frame_save_dir, exist_ok=True)
    
#     # 프레임 저장 (while 루프 안에 추가)
#     frame_time = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
#     frame_filename = os.path.join(frame_save_dir, f"frame_{frame_time}.jpg")
#     cv2.imwrite(frame_filename, frame)

#     # 5. 탐지 수행 (사람 등)
#     det_results = det_model.predict(source=frame, conf=0.4, verbose=False)[0]

#     current_time = time.time()
#     elapsed = current_time - start_time
#     detected_rows = []

#     for box in det_results.boxes:
#         cls_id = int(box.cls[0])
#         name = det_results.names[cls_id]

#         if name != 'person':
#             continue  # 사람만 대상으로 행동 분류

#         x1, y1, x2, y2 = map(int, box.xyxy[0])
#         conf = float(box.conf[0])

#         # 박스 영역 crop → 행동 분류
#         person_img = frame[y1:y2, x1:x2]
#         if person_img.size == 0:
#             continue

#         cls_results = cls_model.predict(person_img, imgsz=224, verbose=False)[0]
#         top1 = int(cls_results.probs.top1)
#         behavior = cls_results.names[top1]
#         behavior_conf = float(cls_results.probs.top1conf)

#         # UI에 표시
#         label = f"{behavior} ({behavior_conf:.2f})"
#         cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
#         cv2.putText(frame, label, (x1, y1 - 10),
#                     cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)

#         # 저장용 정보
#         box_info = f"[{x1},{y1},{x2},{y2}]"
#         detected_rows.append([datetime.now().strftime("%H:%M:%S"), name, behavior, f"{behavior_conf:.2f}", box_info])

#     # 프레임 띄우기
#     cv2.imshow("YOLOv8 Detection + Classification", frame)

#     # 10초마다 CSV 저장
#     if elapsed >= 10:
#         with open(csv_filename, mode='a', newline='') as f:
#             writer = csv.writer(f)
#             writer.writerows(detected_rows)
#         print(f"📁 저장된 객체 수: {len(detected_rows)}개")
#         start_time = current_time  # 타이머 리셋

#     # 종료 키
#     if cv2.waitKey(1) & 0xFF == ord('q'):
#         break

# cap.release()
# cv2.destroyAllWindows()


📁 저장된 객체 수: 1개
📁 저장된 객체 수: 2개
📁 저장된 객체 수: 2개
📁 저장된 객체 수: 2개
📁 저장된 객체 수: 2개
📁 저장된 객체 수: 2개
📁 저장된 객체 수: 2개
📁 저장된 객체 수: 2개
📁 저장된 객체 수: 2개
📁 저장된 객체 수: 1개
📁 저장된 객체 수: 2개
📁 저장된 객체 수: 2개
📁 저장된 객체 수: 2개


KeyboardInterrupt: 

웹캠을 통한 실시간 얼굴인식 코드

In [8]:
from deep_sort_realtime.deepsort_tracker import DeepSort

tracker = DeepSort()
print("✅ DeepSORT 설치 및 초기화 완료")

✅ DeepSORT 설치 및 초기화 완료


In [9]:
import cv2
import os
import time
import csv
from datetime import datetime
from ultralytics import YOLO
import face_recognition
from deep_sort_realtime.deepsort_tracker import DeepSort

# -------------------- 설정 --------------------
KNOWN_FACE_DIR = './known_faces'
CSV_FILENAME = 'detect_log.csv'
DETECTION_MODEL = YOLO('yolov8n.pt')  # 사람 탐지 모델
CLASSIFY_MODEL = YOLO('runs/classify/train/weights/best.pt')  # 행동 분류 모델
CONF_THRESHOLD = 0.4
SAVE_INTERVAL = 10  # 초
MIN_FACE_SIZE = 10  # 얼굴 인식 최소 크기 (픽셀)
# ---------------------------------------------

# 1. 등록된 얼굴 로딩
known_face_encodings = []
known_face_names = []

for filename in os.listdir(KNOWN_FACE_DIR):
    path = os.path.join(KNOWN_FACE_DIR, filename)
    image = face_recognition.load_image_file(path)
    encodings = face_recognition.face_encodings(image)
    if encodings:
        known_face_encodings.append(encodings[0])
        known_face_names.append(os.path.splitext(filename)[0])
    else:
        print(f" 얼굴 인식 실패: {filename}")

# 2. CSV 파일 초기화
if not os.path.exists(CSV_FILENAME):
    with open(CSV_FILENAME, mode='w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(['timestamp', 'face_id', 'action'])

# 3. DeepSORT 초기화
tracker = DeepSort(max_age=30)

# 4. 웹캠 연결
cap = cv2.VideoCapture(0)
start_time = time.time()

while True:
    ret, frame = cap.read()
    if not ret:
        break

    # 사람 탐지
    results = DETECTION_MODEL.predict(frame, conf=CONF_THRESHOLD, classes=[0], verbose=False)
    detections = []

    for box in results[0].boxes:
        x1, y1, x2, y2 = map(int, box.xyxy[0])
        conf = float(box.conf[0])
        detections.append(([x1, y1, x2 - x1, y2 - y1], conf, 'person'))

    # 추적
    tracks = tracker.update_tracks(detections, frame=frame)

    # 10초마다 저장
    current_time = time.time()
    if current_time - start_time >= SAVE_INTERVAL:
        for track in tracks:
            if not track.is_confirmed():
                continue

            track_id = track.track_id
            l, t, r, b = map(int, track.to_ltrb())
            person_crop = frame[t:b, l:r]

            # 유효성 검사
            if person_crop is None or person_crop.size == 0:
                print(f" 잘못된 crop (ID {track_id})")
                continue
            h, w = person_crop.shape[:2]
            if h < MIN_FACE_SIZE or w < MIN_FACE_SIZE:
                print(f" 얼굴 너무 작음 (ID {track_id}, size: {w}x{h})")
                continue

            # 얼굴 인식
            face_locations = face_recognition.face_locations(person_crop)
            if not face_locations:
                print(f" 얼굴 감지 실패 (ID {track_id})")
                continue

            face_encodings = face_recognition.face_encodings(person_crop, face_locations)
            if not face_encodings:
                print(f" 얼굴 인코딩 실패 (ID {track_id})")
                continue

            name = f"Unknown_{track_id}"
            matches = face_recognition.compare_faces(known_face_encodings, face_encodings[0], tolerance=0.45)
            if True in matches:
                name = known_face_names[matches.index(True)]

            # 행동 분류
            result = CLASSIFY_MODEL.predict(person_crop, imgsz=224, verbose=False)
            class_id = result[0].probs.top1
            label = result[0].names[class_id]

            # CSV 저장
            timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
            with open(CSV_FILENAME, mode='a', newline='') as f:
                writer = csv.writer(f)
                writer.writerow([timestamp, name, label])
            print(f"[{timestamp}] {name} → {label} 저장됨 ✅")

        start_time = current_time  # 타이머 리셋

    # 실시간 박스 표시
    for track in tracks:
        if not track.is_confirmed():
            continue
        track_id = track.track_id
        l, t, r, b = map(int, track.to_ltrb())
        cv2.rectangle(frame, (l, t), (r, b), (0, 255, 0), 2)
        cv2.putText(frame, f'ID {track_id}', (l, t - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)

    cv2.imshow("Real-time Action Recognition", frame)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()


TypeError: compute_face_descriptor(): incompatible function arguments. The following argument types are supported:
    1. (self: _dlib_pybind11.face_recognition_model_v1, img: numpy.ndarray[(rows,cols,3),numpy.uint8], face: _dlib_pybind11.full_object_detection, num_jitters: int = 0, padding: float = 0.25) -> _dlib_pybind11.vector
    2. (self: _dlib_pybind11.face_recognition_model_v1, img: numpy.ndarray[(rows,cols,3),numpy.uint8], num_jitters: int = 0) -> _dlib_pybind11.vector
    3. (self: _dlib_pybind11.face_recognition_model_v1, img: numpy.ndarray[(rows,cols,3),numpy.uint8], faces: _dlib_pybind11.full_object_detections, num_jitters: int = 0, padding: float = 0.25) -> _dlib_pybind11.vectors
    4. (self: _dlib_pybind11.face_recognition_model_v1, batch_img: list[numpy.ndarray[(rows,cols,3),numpy.uint8]], batch_faces: list[_dlib_pybind11.full_object_detections], num_jitters: int = 0, padding: float = 0.25) -> _dlib_pybind11.vectorss
    5. (self: _dlib_pybind11.face_recognition_model_v

1차 수정 코드

In [ ]:
# import cv2
# from ultralytics import YOLO
# import os
# from datetime import datetime
# import face_recognition

# # 모델 로딩
# person_model = YOLO('yolov8n.pt')             # 사람 탐지 YOLOv8 모델
# face_model = YOLO('yolov8n-face.pt')          # 얼굴 전용 YOLOv8 모델 (사용자 커스텀 또는 공개 모델)
# classify_model = YOLO('runs/classify/train/weights/best.pt')  # 행동 분류 모델

# # 웹캠 설정
# cap = cv2.VideoCapture(0)
# cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1280)
# cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)

# while True:
#     ret, frame = cap.read()
#     if not ret:
#         break

#     # 1단계: 사람 탐지
#     person_results = person_model(frame, conf=0.4)[0]

#     for person_box in person_results.boxes:
#         x1, y1, x2, y2 = map(int, person_box.xyxy[0])
#         person_crop = frame[y1:y2, x1:x2]

#         if person_crop.shape[0] < 50 or person_crop.shape[1] < 50:
#             continue  # 너무 작은 영역은 제외

#         # 2단계: 얼굴 탐지 (crop 내)
#         face_results = face_model(person_crop, conf=0.3)[0]

#         for face_box in face_results.boxes:
#             fx1, fy1, fx2, fy2 = map(int, face_box.xyxy[0])
#             # 절대 좌표로 보정
#             abs_x1, abs_y1 = x1 + fx1, y1 + fy1
#             abs_x2, abs_y2 = x1 + fx2, y1 + fy2

#             face_crop = frame[abs_y1:abs_y2, abs_x1:abs_x2]

#             # 얼굴 인식
#             face_locations = face_recognition.face_locations(face_crop)
#             face_encodings = face_recognition.face_encodings(face_crop, face_locations)

#             if face_encodings:
#                 # 행동 분류
#                 result = classify_model.predict(face_crop, imgsz=224, verbose=False)
#                 class_id = result[0].probs.top1
#                 label = result[0].names[class_id]

#                 # 시각화
#                 cv2.rectangle(frame, (abs_x1, abs_y1), (abs_x2, abs_y2), (0, 255, 0), 2)
#                 cv2.putText(frame, label, (abs_x1, abs_y1 - 10),
#                             cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)

#     cv2.imshow("YOLOv8 Face Detection + Classification", frame)
#     if cv2.waitKey(1) & 0xFF == ord('q'):
#         break

# cap.release()
# cv2.destroyAllWindows()
